# 0. Импорт и конфигурация

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
RANDOM_STATE = 42

def set_global_seed(seed: int = RANDOM_STATE) -> None:
    np.random.seed(seed)

set_global_seed(RANDOM_STATE)
from sklearn.metrics import classification_report
from decision_tree import CustomDecisionTreeClassifier
from random_forest import CustomRandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

# 1. Обработка данных

## Шаг 1. Считать данные

В этой работе вы используете тот же датасет, что и в предыдущей работе

In [56]:
Student_ID = 31

In [57]:
datasets = [
    (
        'Give Me Some Credit',
        'https://www.kaggle.com/competitions/GiveMeSomeCredit/overview',
        'SeriousDlqin2yrs'
    ),
    (
        'Porto Seguro’s Safe Driver Prediction',
        'https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/overview',
        'target'
    ),
    (
        'Statlog (Shuttle)',
        'https://archive.ics.uci.edu/dataset/148/statlog+shuttle',
        'class'
    ),
    (
        'HTRU2',
        'https://archive.ics.uci.edu/dataset/372/htru2',
        'class'
    ),
    (
        'Bank Marketing',
        'https://archive.ics.uci.edu/dataset/222/bank%2Bmarketing',
        'y'
    ),
]

dataset_id = None if Student_ID is None else Student_ID % len(datasets)
if dataset_id is None:
    print("ОШИБКА! Не указан порядковый номер студента в списке группы.")
else:
    print(f"Информация о датасете '{datasets[dataset_id][0]}' доступна по следующей ссылке: {datasets[dataset_id][1]}")
    print(f"Целевая переменная: {datasets[dataset_id][2]}")

Информация о датасете 'Porto Seguro’s Safe Driver Prediction' доступна по следующей ссылке: https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/overview
Целевая переменная: target


Загрузите данные и считайте их в датафрейм

In [58]:
df = pd.read_csv('data/train.zip', compression='zip')

size_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"{size_mb:.2f} MB")

df = df.sample(frac=0.03, random_state=42)
size_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"{size_mb:.2f} MB")

267.93 MB
8.17 MB


## Шаг 2. Обработка данных

В обработке датасета вы имеете (почти) полную свободу (важно в итоге просто побить бейзлайн).

Обработку и подготовку данных можете взять из предыдущей работы

In [59]:
df_base = df.copy().dropna()
df_base.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
256886,642026,0,4,1,5,1,0,1,0,0,...,7,3,2,3,0,1,0,1,0,0
118785,297043,0,6,2,10,1,0,0,0,0,...,6,3,3,5,0,1,1,0,0,0
56083,140591,0,4,1,9,1,0,0,0,1,...,3,1,0,7,0,0,1,0,0,0
542002,1354540,0,0,1,7,1,4,0,1,0,...,1,1,3,6,1,1,0,0,0,0
349518,873173,0,1,1,3,1,0,1,0,0,...,6,1,5,6,0,1,0,0,0,0


In [60]:
df_processed = df_base.copy().drop_duplicates()

feature_columns = df_processed.columns.drop(['id', 'target'])

for feature in feature_columns:
    df_processed[feature] = (df_processed[feature] - df_processed[feature].mean()) / df_processed[feature].std()

df_processed.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
256886,642026,0,1.062512,-0.539656,0.227019,1.18783,-0.295785,1.247203,-0.597688,-0.442595,...,0.663906,1.301508,-0.513101,-1.644536,-0.374171,0.770878,-1.111937,1.573904,-0.730206,-0.42326
118785,297043,0,2.073716,0.971651,2.079017,1.18783,-0.295785,-0.801749,-0.597688,-0.442595,...,0.235226,1.301508,0.072214,-0.917891,-0.374171,0.770878,0.899281,-0.635327,-0.730206,-0.42326
56083,140591,0,1.062512,-0.539656,1.708618,1.18783,-0.295785,-0.801749,-0.597688,2.259276,...,-1.050812,-0.368252,-1.683730,-0.191245,-0.374171,-1.297150,0.899281,-0.635327,-0.730206,-0.42326
542002,1354540,0,-0.959896,-0.539656,0.967818,1.18783,2.692356,-0.801749,1.673019,-0.442595,...,-1.908171,-0.368252,0.072214,-0.554568,2.672428,0.770878,-1.111937,-0.635327,-0.730206,-0.42326
349518,873173,0,-0.454294,-0.539656,-0.513780,1.18783,-0.295785,1.247203,-0.597688,-0.442595,...,0.235226,-0.368252,1.242843,-0.554568,-0.374171,0.770878,-1.111937,-0.635327,-0.730206,-0.42326


## Шаг 3. Разделение на train/val/test

Разделите датафрейм на фичи и на целевую переменную.

In [61]:
X_base, y_base = df_base.drop(['id', 'target'], axis=1).values, df_base['target'].values
X_processed, y_processed = df_processed.drop(['id', 'target'], axis=1).values, df_processed['target'].values

Разделите датафреймы base и processed каждый на выборки train/val/test (запишите их в переменные ниже)

In [62]:
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)

# Разделяем на train/test (80/20)
X_train_processed, X_test_processed, y_train_processed, y_test_processed = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)

# Для валидационной выборки - берем 25% от train (20% от всех данных)
X_train_processed, X_val_processed, y_train_processed, y_val_processed = train_test_split(
    X_train_processed, y_train_processed, test_size=0.25, random_state=42, stratify=y_train_processed
)

# 2. Классификация

## Шаг 1. Подготовка бейзлайна

На датафрейме `df_base` обучите бейзлайн-модели решающего дерева и случайного леса, используя `sklearn`.

Рекомендуется вычислить метрики, чтобы сравнить их с метриками кастомных моделей. 

In [63]:
dtree_base = DecisionTreeClassifier(
    max_depth=10,         
    min_samples_split=20,   
    min_samples_leaf=10,    
    random_state=42
)

rnd_forest_base = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)


dtree_base.fit(X_train_base, y_train_base)
rnd_forest_base.fit(X_train_base, y_train_base)

y_pred_dt = dtree_base.predict(X_test_base)
y_pred_rf = rnd_forest_base.predict(X_test_base)

## Шаг 2. Обучение кастомных моделей

Импортируйте кастомные модели из .py модулей (рекомедуется держать файлы `models_train.ipynb`, `decision_tree.py` и `random_forest.py` в корне, чтобы не возникло проблем с относительными путями и тестами).

После импорта, обучите модели на датафрейме `df_processed`.

Рекомендуется вычислить метрики, чтобы сравнить их с метриками бейзлайн-моделей.

In [64]:
custom_dt_processed = CustomDecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    random_state=42
)

custom_dt_processed.fit(X_train_processed, y_train_processed)
y_pred_custom_dt = custom_dt_processed.predict(X_test_processed)

In [65]:
custom_rf_processed = CustomRandomForestClassifier(
    n_estimators=5,
    max_depth=15,
    min_samples_split=20,
    random_state=42
)

custom_rf_processed.fit(X_train_processed, y_train_processed)
y_pred_custom_rf = custom_rf_processed.predict(X_test_processed)

## Шаг 3. Финал

После обучения бейзлайна и кастомных моделей, подготовьте файлы `baseline_solution.csv` и `solution.csv`, аналогично файлу `example.csv`. Для этого выполните предсказание на тестовой выборке `X_processed_test`.

Также подготовьте файл `labels.csv` с метками классов из вашей тестовой выборки `y_processed_test`.

In [66]:
import pandas as pd
import numpy as np

# 1. Подготовка labels.csv
labels_df = pd.DataFrame({'target': y_test_processed})
labels_df.to_csv('labels.csv', index=False)

# 2. Создание baseline_solution.csv
# Используем sklearn RandomForest как baseline
baseline_predictions = y_pred_rf
baseline_df = pd.DataFrame({'target': baseline_predictions})
baseline_df.to_csv('baseline_solution.csv', index=False)

# 3. Создание solution.csv
# Используем лучшую кастомную модель
f1_custom_dt = f1_score(y_test_processed, y_pred_custom_dt, average='macro')
f1_custom_rf = f1_score(y_test_processed, y_pred_custom_rf, average='macro')

if f1_custom_rf >= f1_custom_dt:
    solution_predictions = y_pred_custom_rf
else:
    solution_predictions = y_pred_custom_dt

solution_df = pd.DataFrame({'target': solution_predictions})
solution_df.to_csv('solution.csv', index=False)